# Install Library

In [ ]:
!pip install transformers
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 16.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Import Library

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer
from google.colab import drive

# Connect to Drive

In [ ]:
# Path folder tempat file CSV berada
folder_path = "/content/drive/MyDrive/File/CODES/Summarize/4_Summary_Suara_Entertain_50_Base1/Sum_Suara_Entertain_50_Base1"

# Mengambil semua file dengan ekstensi .csv dalam folder
csv_files = [file for file in os.listdir(folder_path) if file.endswith(".csv")]

# Membaca dan menggabungkan semua file
dataframes = []
for file_name in csv_files:
    file_path = os.path.join(folder_path, file_name)
    df = pd.read_csv(file_path)
    dataframes.append(df)

# Menggabungkan semua DataFrame
combined_df = pd.concat(dataframes, ignore_index=True)

In [ ]:
# Menyimpan hasil penggabungan ke file baru
output_path = os.path.join(folder_path, "/content/drive/MyDrive/File/CODES/Similarity/4_Similarity_Suara_Entertain_Base1/suara_nas_ent_50_base1_concat.csv")
combined_df.to_csv(output_path, index=False)

In [ ]:
combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7459 entries, 0 to 7458
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         7459 non-null   object
 1   category      7459 non-null   object
 2   publish_date  7459 non-null   object
 3   author        7459 non-null   object
 4   article_url   7459 non-null   object
 5   content       7459 non-null   object
 6   summary       7459 non-null   object
dtypes: object(7)
memory usage: 408.0+ KB


In [ ]:
combined_df.head()

,title,category,publish_date,author,article_url,content,summary
0,"Tiba-Tiba Jadi Suami Beristri Dua di Film, Ibr...",Entertainment,2024-09-30,Yazir Farouk,https://www.suara.com/entertainment/2024/09/30...,Drama percintaan di film Laut Tengah ternyata ...,Drama percintaan di film Laut Tengah ternyata ...
1,"Rujuk dengan Ira Nandha, Elmer Syaherman Terny...",Entertainment,2024-09-30,Yohanes Endra,https://www.suara.com/entertainment/2024/09/30...,Rumah tangga Elmer Syaherman dan Ira Nandha se...,Rumah tangga Elmer Syaherman dan Ira Nandha se...
2,"Sudah Keluar, Hasil Visum Lolly untuk Buktikan...",Entertainment,2024-09-30,Yazir Farouk,https://www.suara.com/entertainment/2024/09/30...,"Hasil visum lanjutan anak Nikita Mirzani, Laur...","Hasil visum lanjutan anak Nikita Mirzani, Laur..."
3,Putri Marissya Icha Bikin Konten Nyindir Pakai...,Entertainment,2024-09-30,Yohanes Endra,https://www.suara.com/entertainment/2024/09/30...,Marissya Icha belakangan ini disorot karena du...,Marissya Icha belakangan ini disorot karena du...
4,"Kibarkan Bendera Finis MotoGP Mandalika, Maia ...",Entertainment,2024-09-30,Yazir Farouk,https://www.suara.com/entertainment/2024/09/30...,"Momen istri Irwan Mussry, Maia Estianty, mengi...","Momen istri Irwan Mussry, Maia Estianty, mengi..."


In [ ]:
nan_summary = combined_df[combined_df['summary'].isna()]


nan_summary

,title,category,publish_date,author,article_url,content,summary


In [ ]:
combined_df=combined_df.dropna()

# Duplicate


In [ ]:
data = combined_df.copy()

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7459 entries, 0 to 7458
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         7459 non-null   object
 1   category      7459 non-null   object
 2   publish_date  7459 non-null   object
 3   author        7459 non-null   object
 4   article_url   7459 non-null   object
 5   content       7459 non-null   object
 6   summary       7459 non-null   object
dtypes: object(7)
memory usage: 408.0+ KB


# Tokenize

Mencoba kode lain agar tokenize lebih mudah di pahami

# Word Embedding

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# Memuat IndoBERT dan tokenizer
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1", use_fast=True)
model = AutoModel.from_pretrained("indobenchmark/indobert-base-p1")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

In [ ]:
# Fungsi untuk mendapatkan embedding dari IndoBERT
def get_embeddings(input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
    # Mengambil hidden states dari layer terakhir dan rata-rata embeddingsnya
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings

# Fungsi untuk mengonversi tokenized data ke dalam bentuk input_ids dan mendapatkan embedding
def get_input_ids_and_embeddings(data_column):
    input_ids_list = []
    embeddings_list = []

    for text in data_column:
        # Tokenisasi
        tokens = tokenizer(text, padding='max_length', max_length=256, truncation=True, return_tensors="pt")
        input_ids = tokens['input_ids']  # Ambil input_ids
        input_ids_list.append(input_ids)

        # Mendapatkan embeddings
        embedding = get_embeddings(input_ids)
        embeddings_list.append(embedding)

    return input_ids_list, embeddings_list

# Dapatkan input_ids dan embeddings untuk title dan summary
title_input_ids, title_embeddings = get_input_ids_and_embeddings(data['title'])
summary_input_ids, summary_embeddings = get_input_ids_and_embeddings(data['summary'])


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.


model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

## Mengubah embeddings menjadi array numpy

In [ ]:
# Mengubah embeddings menjadi array numpy agar bisa disimpan dalam DataFrame
title_embeddings_array = np.vstack([embedding.numpy() for embedding in title_embeddings])
summary_embeddings_array = np.vstack([embedding.numpy() for embedding in summary_embeddings])

# Membuat DataFrame baru dengan embeddings numerik
data_embeddings = pd.DataFrame({
    'title_embeddings': list(title_embeddings_array),
    'summary_embeddings': list(summary_embeddings_array)
})

# Cosine Similarity

In [ ]:
# hitung cosine similarity pada matrix berita
cosine_sim = cosine_similarity(title_embeddings_array, summary_embeddings_array)
print(cosine_sim)

[[0.1895412  0.18991452 0.31942648 ... 0.8901996  0.6227047  0.7524663 ]
 [0.18490885 0.18900014 0.3200839  ... 0.8894562  0.62221617 0.75227356]
 [0.18342358 0.18436943 0.3213995  ... 0.8851769  0.6196641  0.74897105]
 ...
 [0.18328479 0.1868766  0.31759587 ... 0.88656986 0.6192827  0.74810344]
 [0.18436992 0.18661727 0.31954402 ... 0.8862506  0.6205713  0.7486057 ]
 [0.18124789 0.18543646 0.3180065  ... 0.8884693  0.62278837 0.75308275]]


In [ ]:
# membuat dataframe dari varible cosine similarity dengan baris dan kolom title
cosine_sim_df = pd.DataFrame(cosine_sim, index=data['summary'], columns=data['title'])

print(cosine_sim_df.shape)

(7459, 7459)


## Check Similarity Result

In [ ]:
# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # The index is a summary, and you need to find the corresponding title
    # to access the similarity value. However, since you have both summaries
    # and titles as indices, you can access the similarity directly using
    # the index and the corresponding title from the 'data' DataFrame.

    title = data.loc[data['summary'] == index, 'title'].iloc[0] # Get the title for the current summary
    similarity_value = cosine_sim_df.loc[index, title]  # Access the similarity using both summary and title

    print(f"Title: {title}")
    print(f"Summary: {index}")
    print(f"Cosine Similarity: {similarity_value}\n")
    print("-" * 50)  # Pemisah antar pasangan

Streaming output truncated to the last 5000 lines.
Title: Sakit Hati Dibully, Aurel Hermansyah Luapkan Lewat Lagu
Summary: Aurel Hermansyah sering menjadi sasaran bully warganet, mulai dari fisik hingga tingkahnya yang selalu dianggap salah. Pandangan orang suka 'oh kamu kan artis, ini itu. Padahal sebenarnya sama-sama manusia pasti pernah merasakan capek, kecewa dan sedih," imbuhnya. Atta Halilintar kemudian menciptakan lagu berjudul Aku Juga Manusia. Isinya, memberikan semangat kepada Aurel Hermansyah untuk tak peduli pada ocehan orang. " Bukan hanya ingin Aurel Hermansyah bangkit, Atta Halilintar juga berharap sang istri atau siapapun yang merasa terpuruk bisa meluapkan emosi. Sebab ada penggalan lirik berisi, 'Menangislah, kita manusia. Namun tentunya, hal itu ditujukan kepada orang-orang yang tepat. " Soalnya kalau dipendam akan menjadi bom waktu," terang ibu dua anak ini.
Cosine Similarity: 0.7322392463684082

--------------------------------------------------
Title: Asila Maisa 

In [ ]:
# List to store the results
results = []

# Menampilkan pasangan similarity antara 'title' dan 'summary' yang sesuai dengan urutan
for index, row in cosine_sim_df.iterrows():
    # Mendapatkan judul yang sesuai dengan summary berdasarkan index
    title = data.loc[data['summary'] == index, 'title'].iloc[0]
    similarity_value = cosine_sim_df.loc[index, title]  # Mengakses nilai similarity

    # Menyimpan pasangan dan nilai similarity ke dalam list
    results.append([title, index, similarity_value])

# Membuat DataFrame untuk hasilnya
similarity_df = pd.DataFrame(results, columns=['Title', 'Summary', 'Cosine Similarity'])


In [ ]:
similarity_df.head()

,Title,Summary,Cosine Similarity
0,"Tiba-Tiba Jadi Suami Beristri Dua di Film, Ibr...",Drama percintaan di film Laut Tengah ternyata ...,0.189541
1,"Rujuk dengan Ira Nandha, Elmer Syaherman Terny...",Rumah tangga Elmer Syaherman dan Ira Nandha se...,0.189
2,"Sudah Keluar, Hasil Visum Lolly untuk Buktikan...","Hasil visum lanjutan anak Nikita Mirzani, Laur...",0.3214
3,Putri Marissya Icha Bikin Konten Nyindir Pakai...,Marissya Icha belakangan ini disorot karena du...,0.221442
4,"Kibarkan Bendera Finis MotoGP Mandalika, Maia ...","Momen istri Irwan Mussry, Maia Estianty, mengi...",0.487217


## Save Similarity

In [ ]:
# Menyimpan DataFrame ke file CSV di lokasi yang ditentukan
similarity_df.to_csv('/content/drive/MyDrive/File/CODES/Similarity/4_Similarity_Suara_Entertain_Base1/similarity_suara_ent_base1.csv', index=False)